# Notebook Overview — Generate CLIP Text Representations

## Purpose

This notebook generates reusable CLIP text representations for NExT-QA questions and multiple-choice answer options. These shared text representations provide the semantic question and answer embeddings used by the representation-based VideoQA experiments.

The notebook loads NExT-QA annotation records, constructs normalized text input records for questions and answer choices, loads a pretrained CLIP text encoder, generates normalized text embeddings, validates the resulting representation dataset, and saves the generated artifacts for downstream Fusion MLP classifier experiments.

CLIP text representations are shared across both representation-based pipelines. The same `clip_text` artifacts are used with either `clip_video` representations or `autoencoder_video` representations in Notebook 07.

## Inputs

* Shared project configuration and constants
* NExT-QA annotation files containing questions, answer choices, and ground-truth labels
* CLIP text dataset mode configuration controlling development-subset or full-dataset generation
* Pretrained CLIP text encoder model

## Outputs

* CLIP text representation dataset
* CLIP text representation summary report
* Representation validation results
* Sample representation records
* Shared Google Drive artifacts

## Processing Workflow

1. Initialize the notebook environment.
2. Configure CLIP text representation generation.
3. Verify the runtime environment.
4. Prepare text input records.
5. Load the pretrained CLIP text encoder.
6. Generate CLIP text representations.
7. Validate the generated representation dataset.
8. Save representation artifacts.
9. Generate representation summary reports.
10. Display representative text representation records.
11. Summarize notebook outputs and generated artifacts.

## Downstream Consumer

Notebook 07 — Run Representation-Based VideoQA



### 🔷 Step 1 — Initialize Environment for CLIP Text Representation Generation

* Configure notebook execution controls for development-subset or full-dataset generation.
* Mount Google Drive and prepare the Colab execution environment.
* Clone the project repository and load shared configuration constants and utility modules.
* Verify required project paths and output directories.
* Load NExT-QA annotation metadata required for text representation generation.
* Prepare the notebook environment for CLIP-based text representation generation.


In [ ]:
# ============================================================
# Step 1: Initialize Environment for CLIP Text Representation Generation
# ============================================================

# ============================================================
# Notebook Execution Controls
# ============================================================

# False -> Generate embeddings for the development subset.
# True  -> Generate embeddings for the complete NExT-QA dataset.
GENERATE_FULL_DATASET = True

VERBOSE = True
REQUIRE_L4_GPU = True

import os
import time
from pathlib import Path

import pandas as pd

from google.colab import drive, userdata

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# CLONE REPOSITORY
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://x-access-token:{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD PROJECT MODULES
# ------------------------------------------------------------

print("\nLoading project configuration and utility modules...")

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# OUTPUT SETUP
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = [
    Path("src"),
    Path("datasets"),
    OUTPUTS_DIR,
    QUESTIONS_DIR,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# LOAD NExT-QA ANNOTATIONS
# ------------------------------------------------------------

print("\nLoading NExT-QA annotations...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata loaded.")
print(f"Annotation records: {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook ready for CLIP text representation generation.")



### 🔷 Step 2 — Define CLIP Text Representation Configuration

* Define the shared CLIP text representation configuration used throughout the notebook.
* Configure development-subset or full-dataset representation generation.
* Specify the active representation source, evaluation split, development subset size, and randomization settings.
* Configure the pretrained CLIP text encoder used to generate `clip_text` representations.
* Define the shared output artifact locations.
* Display the active representation configuration for the current experiment.


In [ ]:
# ============================================================
# Step 2: Define CLIP Text Representation Configuration
# ============================================================

print("Defining CLIP text representation configuration...")

# ------------------------------------------------------------
# CLIP text model configuration
# ------------------------------------------------------------

CLIP_TEXT_MODEL_NAME = "openai/clip-vit-base-patch32"

TEXT_REPRESENTATION_SCOPE = CLIP_TEXT_REPRESENTATION_SCOPE

TEXT_INPUT_TYPES = [
    "question",
    "answer_choice",
]

QUESTION_TEXT_FIELD = QUESTION_COLUMN
ANSWER_CHOICE_COLUMNS = CHOICE_COLUMNS

# ------------------------------------------------------------
# Dataset generation configuration
# ------------------------------------------------------------

evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

clip_text_config_summary = {
    "artifact_scope": "shared",
    "clip_text_model": CLIP_TEXT_MODEL_NAME,
    "representation_scope": TEXT_REPRESENTATION_SCOPE,
    "text_input_types": ", ".join(TEXT_INPUT_TYPES),
    "generate_full_dataset": GENERATE_FULL_DATASET,
    "evaluation_split": evaluation_split,
    "development_subset_size": development_subset_size,
    "random_seed": random_seed,
    "question_text_field": QUESTION_TEXT_FIELD,
    "answer_choice_columns": ", ".join(ANSWER_CHOICE_COLUMNS),
    "output_directory": str(CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR),
}

clip_text_config_df = pd.DataFrame(
    clip_text_config_summary.items(),
    columns=["Configuration Item", "Value"],
)

print("CLIP text representation configuration defined.")
display(clip_text_config_df)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify the active Python runtime, operating system, and PyTorch installation.
* Confirm that CUDA is available and validate the required NVIDIA L4 GPU when enabled.
* Display detected GPU hardware and available GPU memory.
* Verify that the Hugging Face Transformers library is installed and available.
* Confirm that the required CLIP model and processor classes can be imported successfully.
* Validate that the runtime environment satisfies all software and hardware requirements.
* Display a runtime verification summary before loading the CLIP text model.

In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import platform
import sys

import torch

print("Verifying runtime environment...")
print("-" * 60)

# ------------------------------------------------------------
# Python
# ------------------------------------------------------------

print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------

print(f"PyTorch Version: {torch.__version__}")

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

cuda_available = torch.cuda.is_available()

print(f"CUDA Available : {cuda_available}")

if REQUIRE_L4_GPU:

    if not cuda_available:
        raise RuntimeError(
            "CUDA GPU is required for CLIP text representation generation."
        )

    gpu_name = torch.cuda.get_device_name(0)

    print(f"GPU            : {gpu_name}")

    if "L4" not in gpu_name:
        raise RuntimeError(
            f"NVIDIA L4 GPU required. Detected: {gpu_name}"
        )

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory_gb = (
        gpu_properties.total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory     : {gpu_memory_gb:.1f} GB")

# ------------------------------------------------------------
# Transformers
# ------------------------------------------------------------

try:
    import transformers

    print(f"Transformers   : {transformers.__version__}")

except ImportError:

    raise ImportError(
        "The transformers package is required."
    )

# ------------------------------------------------------------
# Verify CLIP Classes
# ------------------------------------------------------------

try:
    from transformers import (
        CLIPModel,
        CLIPProcessor,
    )

    print("CLIP classes   : Available")

except ImportError:

    raise ImportError(
        "Unable to import CLIPModel and CLIPProcessor."
    )

# ------------------------------------------------------------
# Runtime Summary
# ------------------------------------------------------------

print("\nRuntime verification complete.")
print("-" * 60)
print("Environment is ready for CLIP text representation generation.")



### 🔷 Step 4 — Prepare CLIP Text Input Dataset

* Select either the configured development subset or the complete NExT-QA annotation dataset for CLIP text representation generation.
* Validate the required question, answer, and answer-choice fields prior to text representation processing.
* Generate a reproducible development subset when development execution mode is selected.
* Associate each question with its corresponding ground-truth answer text.
* Construct normalized text input records for both questions and multiple-choice answer options.
* Attach representation metadata required for downstream CLIP text embedding generation.
* Validate the generated text input dataset prior to CLIP text embedding generation.
* Display summary statistics and representative text input records for verification.

In [ ]:
# ============================================================
# Step 4: Prepare CLIP Text Input Dataset
# ============================================================

import pandas as pd

print("Preparing CLIP text input dataset...")

# ------------------------------------------------------------
# Validate annotation columns
# ------------------------------------------------------------

required_annotation_columns = [
    "split",
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
    *CHOICE_COLUMNS,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

# ------------------------------------------------------------
# Select annotation records
# ------------------------------------------------------------

if GENERATE_FULL_DATASET:

    eval_df = (
        annotations_df
        .copy()
        .reset_index(drop=True)
    )

else:

    eval_df = annotations_df[
        annotations_df["split"] == evaluation_split
    ].copy()

    if len(eval_df) == 0:
        raise ValueError(f"No records found for split: {evaluation_split}")

    sample_size = min(
        development_subset_size,
        len(eval_df),
    )

    eval_df = (
        eval_df
        .sample(
            n=sample_size,
            random_state=random_seed,
        )
        .reset_index(drop=True)
    )

if len(eval_df) == 0:
    raise RuntimeError("No annotation records were selected.")

eval_df[VIDEO_ID_COLUMN] = eval_df[VIDEO_ID_COLUMN].astype(str)

input_splits = sorted(
    eval_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print(f"Dataset mode              : {dataset_mode_label}")
print(f"Selected annotation rows  : {len(eval_df):,}")
print(f"Input splits              : {', '.join(input_splits)}")

# ------------------------------------------------------------
# Attach ground-truth answer text
# ------------------------------------------------------------

def answer_index_to_text(row):
    answer_idx = int(row[GROUND_TRUTH_ANSWER_COLUMN])
    option_col = f"a{answer_idx}"

    if option_col not in CHOICE_COLUMNS:
        raise ValueError(
            f"Answer option column not valid: {option_col}"
        )

    return row[option_col]

eval_df["ground_truth_text"] = eval_df.apply(
    answer_index_to_text,
    axis=1,
)

# ------------------------------------------------------------
# Build normalized text input records
# ------------------------------------------------------------

text_records = []

for row_index, row in eval_df.iterrows():

    video_id = str(row[VIDEO_ID_COLUMN])

    raw_question_id = row.get(
        "question_id",
        row.get("qid", None),
    )

    if pd.isna(raw_question_id):
        question_id = row_index
    else:
        question_id = raw_question_id

    annotation_id = (
        f"{row['split']}_{video_id}_{question_id}_{row_index}"
    )

    # Question text record
    text_records.append(
        {
            "record_id": f"{annotation_id}_question",
            "video": video_id,
            "question_id": question_id,
            "annotation_id": annotation_id,
            "text_type": "question",
            "choice_index": None,
            "text": row[QUESTION_COLUMN],
            "answer": int(row[GROUND_TRUTH_ANSWER_COLUMN]),
            "ground_truth_text": row["ground_truth_text"],
            "split": row["split"],
            "representation_source": "clip_text",
        }
    )

    # Answer-choice text records
    for choice_index, choice_col in enumerate(CHOICE_COLUMNS):
        text_records.append(
            {
                "record_id": f"{annotation_id}_choice_{choice_index}",
                "video": video_id,
                "question_id": question_id,
                "annotation_id": annotation_id,
                "text_type": "answer_choice",
                "choice_index": choice_index,
                "text": row[choice_col],
                "answer": int(row[GROUND_TRUTH_ANSWER_COLUMN]),
                "ground_truth_text": row["ground_truth_text"],
                "split": row["split"],
                "representation_source": "clip_text",
            }
        )

text_input_df = pd.DataFrame(text_records)

if text_input_df.empty:
    raise RuntimeError("No CLIP text input records were generated.")

# ------------------------------------------------------------
# Validate generated text inputs
# ------------------------------------------------------------

required_text_columns = [
    "record_id",
    "video",
    "question_id",
    "annotation_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "split",
    "representation_source",
]

missing_text_columns = [
    col for col in required_text_columns
    if col not in text_input_df.columns
]

if missing_text_columns:
    raise ValueError(
        f"text_input_df is missing required columns: {missing_text_columns}"
    )

empty_text_count = (
    text_input_df["text"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

if empty_text_count > 0:
    raise ValueError(
        f"Found {empty_text_count} empty text input records."
    )

duplicate_record_id_count = (
    text_input_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_record_id_count > 0:
    raise ValueError(
        f"Found {duplicate_record_id_count} duplicate record_id values."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

text_type_summary_df = (
    text_input_df
    .groupby("text_type")
    .size()
    .reset_index(name="record_count")
)

print("\nCLIP text input dataset prepared successfully.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Annotation records : {len(eval_df):,}")
print(f"Text input records : {len(text_input_df):,}")
print(f"Unique videos      : {eval_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Input splits       : {', '.join(input_splits)}")
print(f"Answer mode        : {ANSWER_MODE}")

print("\nText Type Summary:")
display(text_type_summary_df)

print("\nText Input Preview (First Question):")

first_annotation_id = text_input_df["annotation_id"].iloc[0]

display(
    text_input_df[
        text_input_df["annotation_id"] == first_annotation_id
    ]
)



### 🔷 Step 5 — Load CLIP Text Model

* Select the appropriate computation device for CLIP text representation generation.
* Load the pretrained CLIP text encoder model from the Hugging Face Transformers library.
* Load the corresponding CLIP processor used for text tokenization and preprocessing.
* Configure the CLIP model for inference by switching to evaluation mode.
* Verify the text embedding dimension and maximum supported text sequence length.
* Perform a sample inference to confirm successful text embedding generation.
* Display model configuration details and verification results prior to batch representation generation.

In [ ]:
# ============================================================
# Step 5: Load CLIP Text Model
# ============================================================

import torch

from transformers import (
    CLIPModel,
    CLIPProcessor,
)

print("Loading CLIP text model...")

# ------------------------------------------------------------
# Select computation device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {device}")

# ------------------------------------------------------------
# Load CLIP model
# ------------------------------------------------------------

clip_model = CLIPModel.from_pretrained(
    CLIP_TEXT_MODEL_NAME
)

clip_model.to(device)
clip_model.eval()

# ------------------------------------------------------------
# Load CLIP processor
# ------------------------------------------------------------

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_TEXT_MODEL_NAME
)

# ------------------------------------------------------------
# Verify model configuration
# ------------------------------------------------------------

text_embedding_dimension = (
    clip_model.config.projection_dim
)

text_max_position_embeddings = (
    clip_model.text_model.config.max_position_embeddings
)

print("\nCLIP text model loaded successfully.")
print(f"Model                     : {CLIP_TEXT_MODEL_NAME}")
print(f"Embedding dimension       : {text_embedding_dimension}")
print(f"Maximum text length       : {text_max_position_embeddings}")
print(f"Model device              : {device}")

# ------------------------------------------------------------
# Verify inference
# ------------------------------------------------------------

sample_inputs = clip_processor(
    text=["CLIP model verification"],
    return_tensors="pt",
    padding=True,
)

sample_inputs = {
    key: value.to(device)
    for key, value in sample_inputs.items()
}

with torch.no_grad():
    sample_outputs = clip_model.text_model(
        **sample_inputs
    )

    sample_features = sample_outputs.pooler_output

print(
    f"Verification embedding shape : "
    f"{tuple(sample_features.shape)}"
)

print("\nCLIP text model is ready for representation generation.")



### 🔷 Step 6 — Generate CLIP Text Representations

* Generate CLIP text embeddings for each prepared question and multiple-choice answer option.
* Process text inputs in batches to improve inference performance and GPU utilization.
* Tokenize and encode text using the pretrained CLIP text encoder.
* Normalize embedding vectors to produce consistent representation magnitudes.
* Associate each embedding with its corresponding text record and representation metadata.
* Construct the complete CLIP text representation dataset for downstream VideoQA experiments.
* Verify that the expected embedding dimensions were generated successfully.
* Display summary statistics describing the completed text representation generation process.


In [ ]:
# ============================================================
# Step 6: Generate CLIP Text Representations
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

print("Generating CLIP text representations...")

if "text_input_df" not in globals():
    raise NameError("text_input_df was not found. Run Step 4 first.")

if "clip_model" not in globals():
    raise NameError("clip_model was not found. Run Step 5 first.")

if "clip_processor" not in globals():
    raise NameError("clip_processor was not found. Run Step 5 first.")

# ------------------------------------------------------------
# Encoding configuration
# ------------------------------------------------------------

clip_model.eval()

records = []
start_time = time.time()

# ------------------------------------------------------------
# Batch text encoding
# ------------------------------------------------------------

for start_idx in tqdm(
    range(0, len(text_input_df), CLIP_TEXT_BATCH_SIZE),
    desc="Encoding CLIP text",
):

    batch_df = text_input_df.iloc[
        start_idx:start_idx + CLIP_TEXT_BATCH_SIZE
    ].copy()

    batch_texts = (
        batch_df["text"]
        .astype(str)
        .tolist()
    )

    inputs = clip_processor(
        text=batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        text_outputs = clip_model.text_model(
            **inputs
        )

        pooled_output = text_outputs.pooler_output

        text_features = clip_model.text_projection(
            pooled_output
        )

        text_features = text_features / text_features.norm(
            dim=-1,
            keepdim=True,
        )

    text_features_np = (
        text_features
        .detach()
        .cpu()
        .numpy()
    )

    for row, embedding in zip(
        batch_df.to_dict("records"),
        text_features_np,
    ):

        record = dict(row)

        for i, value in enumerate(embedding):
            record[f"clip_text_{i:03d}"] = float(value)

        records.append(record)

elapsed_time = time.time() - start_time

clip_text_representation_df = pd.DataFrame(records)

if clip_text_representation_df.empty:
    raise RuntimeError(
        "No CLIP text representations were generated."
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

clip_text_columns = [
    col
    for col in clip_text_representation_df.columns
    if col.startswith("clip_text_")
]

if len(clip_text_columns) == 0:
    raise ValueError(
        "No CLIP text embedding columns were generated."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print("\nCLIP text representation generation complete.")
print(f"Dataset mode         : {dataset_mode_label}")
print(f"Input text records   : {len(text_input_df):,}")
print(f"Embedding records    : {len(clip_text_representation_df):,}")
print(f"Embedding dimensions : {len(clip_text_columns):,}")
print(f"Batch size           : {CLIP_TEXT_BATCH_SIZE}")
print(f"Elapsed time         : {elapsed_time:.1f} seconds")
print(
    f"Average/record       : "
    f"{elapsed_time / len(clip_text_representation_df):.3f} seconds"
)



### 🔷 Step 7 — Validate CLIP Text Representation Dataset

* Verify that the CLIP text representation dataset was generated successfully.
* Validate the number of question records, answer-choice records, videos, and unique question records.
* Confirm that the expected CLIP text embedding dimensions were produced.
* Verify that all embedding values are present and numeric.
* Check for duplicate representation records within the generated dataset.
* Confirm that representation metadata was propagated correctly to all representation records.
* Display a validation summary describing the integrity of the generated CLIP text representation dataset.
* Verify that the representation dataset is ready for downstream VideoQA experiments.

In [ ]:
# ============================================================
# Step 7: Validate CLIP Text Representation Dataset
# ============================================================

import pandas as pd

print("Validating CLIP text representation dataset...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Validation Summary
# ------------------------------------------------------------

validation_summary = {
    "representation_records": len(
        clip_text_representation_df
    ),
    "question_records": (
        clip_text_representation_df["text_type"]
        .eq("question")
        .sum()
    ),
    "answer_choice_records": (
        clip_text_representation_df["text_type"]
        .eq("answer_choice")
        .sum()
    ),
    "unique_videos": (
        clip_text_representation_df["video"]
        .nunique()
    ),
    "unique_question_records": (
        clip_text_representation_df
        .loc[
            clip_text_representation_df["text_type"] == "question",
            "annotation_id"
        ]
        .nunique()
    ),
    "embedding_dimensions": len(
        clip_text_columns
    ),
    "missing_embedding_values": (
        clip_text_representation_df[
            clip_text_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
    "non_numeric_embedding_columns": sum(
        not pd.api.types.is_numeric_dtype(
            clip_text_representation_df[col]
        )
        for col in clip_text_columns
    ),
    "duplicate_record_ids": (
        clip_text_representation_df["record_id"]
        .duplicated()
        .sum()
    ),
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Value"],
)

display(validation_df)

# ------------------------------------------------------------
# Validation Checks
# ------------------------------------------------------------

if validation_summary["missing_embedding_values"] > 0:
    raise ValueError(
        "Missing values detected in CLIP text representations."
    )

if validation_summary["non_numeric_embedding_columns"] > 0:
    raise ValueError(
        "Non-numeric embedding columns detected."
    )

if validation_summary["duplicate_record_ids"] > 0:
    raise ValueError(
        "Duplicate record IDs detected."
    )

print(
    "\nRepresentation validation passed. "
    "No missing, non-numeric, or duplicate records detected."
)



### 🔷 Step 8 — Save CLIP Text Representation Files

* Create the local output directory when necessary.
* Save the generated CLIP text representation dataset to local project storage.
* Verify that the representation dataset was written successfully.
* Copy the representation dataset to the shared Google Drive output directory when full-dataset generation is selected.
* Report the number of generated representation records and embedding dimensions.
* Display the output locations for downstream representation-based VideoQA experiments.



In [ ]:
# ============================================================
# Step 8: Save CLIP Text Representation Files
# ============================================================

import shutil

print("Saving CLIP text representation files...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Determine dataset mode
# ------------------------------------------------------------

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

# ------------------------------------------------------------
# Create local output directory
# ------------------------------------------------------------

CLIP_TEXT_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Save representation dataset locally
# ------------------------------------------------------------

clip_text_representation_df.to_csv(
    CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
    index=False,
)

# ------------------------------------------------------------
# Verify local output
# ------------------------------------------------------------

if not CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create local file: "
        f"{CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV}"
    )

# ------------------------------------------------------------
# Copy full-dataset artifact to Google Drive
# ------------------------------------------------------------

drive_artifact_written = False

if GENERATE_FULL_DATASET:

    CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
        CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV,
    )

    if not CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV.exists():
        raise FileNotFoundError(
            f"Failed to create Drive file: "
            f"{CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}"
        )

    drive_artifact_written = True

else:

    print(
        "Development mode complete. Local representation file was "
        "created, but the persistent shared Drive artifact was not "
        "overwritten."
    )

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nCLIP text representation dataset saved successfully.")
print(f"Dataset mode         : {dataset_mode_label}")
print(f"Local output file    : {CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV}")

if drive_artifact_written:
    print(f"Drive output file    : {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}")
else:
    print("Drive output file    : not written in development mode")

print(f"Representation rows  : {len(clip_text_representation_df):,}")
print(f"Embedding dimensions : {len(clip_text_columns):,}")
print(
    f"Question records     : "
    f"{(clip_text_representation_df['text_type'] == 'question').sum():,}"
)
print(
    f"Answer-choice records: "
    f"{(clip_text_representation_df['text_type'] == 'answer_choice').sum():,}"
)



### 🔷 Step 9 — Generate CLIP Text Representation Summary Report

* Generate summary statistics describing the completed CLIP text representation dataset.
* Summarize the number of generated question and answer-choice representations.
* Report the number of unique videos, unique question records, and input dataset splits included in the generated representation dataset.
* Verify the generated CLIP text embedding dimensionality and check for missing embedding values.
* Record shared representation metadata required for downstream representation-based VideoQA experiments.
* Save the representation summary report to local project storage.
* Copy the summary report to the shared Google Drive output directory when full-dataset generation is selected.
* Display the completed CLIP text representation summary for verification.


In [ ]:
# ============================================================
# Step 9: Generate CLIP Text Representation Summary Report
# ============================================================

import shutil
import pandas as pd

print("Generating CLIP text representation summary report...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Determine dataset mode
# ------------------------------------------------------------

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

input_splits = sorted(
    clip_text_representation_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# Compute summary statistics
# ------------------------------------------------------------

question_record_count = (
    clip_text_representation_df["text_type"]
    .eq("question")
    .sum()
)

answer_choice_record_count = (
    clip_text_representation_df["text_type"]
    .eq("answer_choice")
    .sum()
)

unique_question_records = (
    clip_text_representation_df
    .loc[
        clip_text_representation_df["text_type"] == "question",
        "annotation_id"
    ]
    .nunique()
)

missing_embedding_values = (
    clip_text_representation_df[clip_text_columns]
    .isna()
    .sum()
    .sum()
)

summary_rows = [
    {"metric": "artifact_scope", "value": "shared"},
    {"metric": "dataset_mode", "value": dataset_mode_label},
    {"metric": "input_splits", "value": ", ".join(input_splits)},
    {"metric": "representation_type", "value": "clip_text_representation"},
    {"metric": "clip_text_model", "value": CLIP_TEXT_MODEL_NAME},
    {"metric": "clip_text_batch_size", "value": CLIP_TEXT_BATCH_SIZE},
    {"metric": "representation_scope", "value": CLIP_TEXT_REPRESENTATION_SCOPE},
    {"metric": "answer_mode", "value": ANSWER_MODE},
    {"metric": "representation_records", "value": len(clip_text_representation_df)},
    {"metric": "question_records", "value": int(question_record_count)},
    {"metric": "answer_choice_records", "value": int(answer_choice_record_count)},
    {"metric": "unique_videos", "value": clip_text_representation_df["video"].nunique()},
    {"metric": "unique_question_records", "value": int(unique_question_records)},
    {"metric": "embedding_dimensions", "value": len(clip_text_columns)},
    {"metric": "missing_embedding_values", "value": int(missing_embedding_values)},
]

clip_text_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report locally
# ------------------------------------------------------------

CLIP_TEXT_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

clip_text_summary_df.to_csv(
    CLIP_TEXT_SUMMARY_LOCAL_CSV,
    index=False,
)

if not CLIP_TEXT_SUMMARY_LOCAL_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create local summary file: "
        f"{CLIP_TEXT_SUMMARY_LOCAL_CSV}"
    )

# ------------------------------------------------------------
# Copy full-dataset summary artifact to Google Drive
# ------------------------------------------------------------

summary_drive_artifact_written = False

if GENERATE_FULL_DATASET:

    CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CLIP_TEXT_SUMMARY_LOCAL_CSV,
        CLIP_TEXT_SUMMARY_DRIVE_CSV,
    )

    if not CLIP_TEXT_SUMMARY_DRIVE_CSV.exists():
        raise FileNotFoundError(
            f"Failed to create Drive summary file: "
            f"{CLIP_TEXT_SUMMARY_DRIVE_CSV}"
        )

    summary_drive_artifact_written = True

else:

    print(
        "Development mode complete. Local summary report was "
        "created, but the persistent shared Drive summary was not "
        "overwritten."
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("CLIP text representation summary report saved.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Local summary file : {CLIP_TEXT_SUMMARY_LOCAL_CSV}")

if summary_drive_artifact_written:
    print(f"Drive summary file : {CLIP_TEXT_SUMMARY_DRIVE_CSV}")
else:
    print("Drive summary file : not written in development mode")

display(clip_text_summary_df)



### 🔷 Step 10 — Display Sample CLIP Text Representation Records

* Randomly select representative CLIP text representation records for inspection.
* Display representation metadata together with selected embedding information.
* Verify that question and answer-choice representations were generated correctly.
* Provide a qualitative sanity check before downstream representation-based VideoQA experiments.



In [ ]:
# ============================================================
# Step 10: Validate CLIP Text Representation Dataset
# ============================================================

import pandas as pd

print("Displaying sample CLIP text representation records...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Select sample records
# ------------------------------------------------------------

sample_count = min(
    10,
    len(clip_text_representation_df),
)

sample_text_representation_df = (
    clip_text_representation_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

display_columns = [
    "record_id",
    "video",
    "question_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "representation_source",
    *clip_text_columns[:5],
]

print(f"Displaying {sample_count} CLIP text representation records...")

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)
print(f"Dataset mode          : {dataset_mode_label}")

print(
    f"Input splits          : "
    f"{', '.join(sorted(clip_text_representation_df['split'].astype(str).unique()))}"
)

print(f"CLIP model            : {CLIP_TEXT_MODEL_NAME}")
print(f"Representation source : clip_text")
print(f"Embedding dimensions  : {len(clip_text_columns)}")
print("Showing 10 sample records with the first 5 embedding columns.")

display(
    sample_text_representation_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Representation Summary
# ------------------------------------------------------------

question_record_count = (
    clip_text_representation_df["text_type"]
    .eq("question")
    .sum()
)

answer_choice_record_count = (
    clip_text_representation_df["text_type"]
    .eq("answer_choice")
    .sum()
)

unique_question_records = (
    clip_text_representation_df
    .loc[
        clip_text_representation_df["text_type"] == "question",
        "annotation_id"
    ]
    .nunique()
)

print("\nCLIP Text Representation Summary")
print("-" * 60)
print(f"Representation records   : {len(clip_text_representation_df):,}")
print(f"Question records         : {question_record_count:,}")
print(f"Answer-choice records    : {answer_choice_record_count:,}")
print(f"Unique videos            : {clip_text_representation_df['video'].nunique():,}")
print(f"Unique question records  : {unique_question_records:,}")
print(f"Embedding dimensions     : {len(clip_text_columns):,}")



### 🔷 Step 11 — Notebook Summary

* Summarize the completed CLIP text representation generation workflow.
* Report the active representation configuration, dataset mode, CLIP model, and generated representation statistics.
* List the generated CLIP text representation artifacts.
* Confirm that the generated `clip_text` representations are ready for downstream Fusion MLP classification in Notebook 07.


In [ ]:
# ============================================================
# Step 11: Notebook Summary
# ============================================================

print("Notebook 05 complete.")
print("=" * 60)

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

input_splits = sorted(
    clip_text_representation_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

print("\nCLIP Text Representations — Shared Configuration")
print("-" * 60)
print("Artifact scope           : shared")
print(f"Dataset mode             : {dataset_mode_label}")
print(f"CLIP text model          : {CLIP_TEXT_MODEL_NAME}")
print(f"Input splits             : {', '.join(input_splits)}")

if GENERATE_FULL_DATASET:
    print("Development subset size  : not applicable")
else:
    print(f"Development subset size  : {DEVELOPMENT_SUBSET_SIZE}")

print(f"QA format                : {ANSWER_MODE}")

print("\nRepresentation Dataset Summary")
print("-" * 60)
print(f"Representation records   : {len(clip_text_representation_df):,}")

print(
    f"Question records         : "
    f"{(clip_text_representation_df['text_type'] == 'question').sum():,}"
)

print(
    f"Answer-choice records    : "
    f"{(clip_text_representation_df['text_type'] == 'answer_choice').sum():,}"
)

print(f"Unique videos            : {clip_text_representation_df['video'].nunique():,}")

unique_question_records: (
    clip_text_representation_df
    .loc[
        clip_text_representation_df["text_type"] == "question",
        "annotation_id"
    ]
    .nunique()
)

print(f"Unique question records  : {unique_question_records:,}")
print(f"Embedding dimensions     : {len(clip_text_columns):,}")

print("\nShared Representation Outputs")
print("-" * 60)
print(f"Local text artifact      : {CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV}")
print(f"Local summary artifact   : {CLIP_TEXT_SUMMARY_LOCAL_CSV}")

if GENERATE_FULL_DATASET:
    print(f"Shared text artifact     : {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}")
    print(f"Shared summary artifact  : {CLIP_TEXT_SUMMARY_DRIVE_CSV}")
else:
    print("Shared text artifact     : not written in development mode")
    print("Shared summary artifact  : not written in development mode")

print(f"Shared output directory  : {CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR}")

print("\nNotebook 05 generated:")
print("- CLIP text representation dataset")
print("- CLIP text representation summary")
print("- Validated representation records")
print("- Sample representation records")

print("\nNotebook 05 outputs are ready for downstream")
print("representation-based VideoQA experiments.")

